<a href="https://colab.research.google.com/github/NizarDag/llm-post-training-labs/blob/main/01_base_vs_sft_vs_rl_gsm8k/01_base_vs_sft_vs_rl_gsm8k.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Comparing Base, SFT, and RL-Tuned Models on GSM8K

## Objective

Compare how post-training changes model behavior on mathematical reasoning tasks.

## Concepts

- Pre-training
- Supervised Fine-Tuning (SFT)
- Reinforcement Learning (RL)
- Reasoning
- GSM8K

## Dataset

GSM8K

## Models

- Base model
- SFT model
- RL model

## Results

Work in progress.

## Experiment Setup

### Research Question

How does post-training affect mathematical reasoning performance?

Specifically, we compare:

1. A Base Model (pre-trained only)
2. A Supervised Fine-Tuned (SFT) Model
3. A Reinforcement Learning (RL) Model

on GSM8K mathematical reasoning tasks.

### Expected Outcomes

- The Base Model may struggle with instruction following and reasoning.
- The SFT Model should follow instructions more reliably.
- The RL Model should demonstrate stronger reasoning behavior and higher accuracy.

In [1]:
!pip install -q datasets transformers accelerate huggingface_hub bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.4 MB/s eta 0:00:00


In [2]:
from datasets import load_dataset

gsm8k_dataset = load_dataset("openai/gsm8k", "main")
gsm8k_dataset

README.md:   0%|          | 0.00/7.93k [00:00<?, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['question', 'answer'],
        num_rows: 7473
    })
    test: Dataset({
        features: ['question', 'answer'],
        num_rows: 1319
    })
})

In [3]:
example = gsm8k_dataset["train"][0]
print("Question:")
print(example["question"])

print("\nAnswer:")
print(example["answer"])

Question:
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?

Answer:
Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
#### 72


## Dataset Exploration

GSM8K (Grade School Math 8k) contains mathmatical work promblems and detailed reasoning solutions.

Each example contains:

- question
- answer

The answer filed contains both the reasoning process and the final answer.

In [4]:
import re

def extract_gsm8k_answer(answer_text):
  match = re.search(r"####\s*(-?\d+\.?\d*)", answer_text)

  if match:
    return match.group(1)

  return None

In [5]:
golden_answer = extract_gsm8k_answer(example["answer"])
print(golden_answer)

72


In [6]:
for i in range(3):
  print("="* 80)
  print("Question:")
  print(gsm8k_dataset['test'][i]['question'])

  print("\nFinal Answer:")
  print(extract_gsm8k_answer(gsm8k_dataset['test'][i]['answer']))

Question:
Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?

Final Answer:
18
Question:
A robe takes 2 bolts of blue fiber and half that much white fiber.  How many bolts in total does it take?

Final Answer:
3
Question:
Josh decides to try flipping a house.  He buys a house for $80,000 and then puts in $50,000 in repairs.  This increased the value of the house by 150%.  How much profit did he make?

Final Answer:
70000


In [ ]:
!nvidia-smi

Thu Jun 18 17:09:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [7]:
model_names = {
    "base": "deepseek-ai/deepseek-math-7b-base",
    "sft": "deepseek-ai/deepseek-math-7b-instruct",
    "rl": "deepseek-ai/deepseek-math-7b-rl"
}

In [8]:
from transformers import AutoTokenizer

In [9]:
for name, model_id in model_names.items():
  print(f"testing {name}: {model_id}")

  try:
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    print("Found")

  except Exception as e:
    print("Failed")
    print(e)

    print("-" * 50)

testing base: deepseek-ai/deepseek-math-7b-base


config.json:   0%|          | 0.00/594 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/4.61M [00:00<?, ?B/s]

Found
testing sft: deepseek-ai/deepseek-math-7b-instruct


...

Found
testing rl: deepseek-ai/deepseek-math-7b-rl


config.json:   0%|          | 0.00/626 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.09k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/4.61M [00:00<?, ?B/s]

Found


## Model Verification

The follwoing models were successfully located on Hugging Face:
- deepseek-ai/deepseek-math-7b-base
- deepseek-ai/deepseek-math-7b-instruct
- deepseek-ai/deepseek-math-7b-rl

These represent three stages of the LLM training pipline:

1. Base Model (pretraining)
2. Supervised Fine-tuned Model (SFT)
3. Reinforcement Learning Model (RL)

In [10]:
import torch
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype = torch.float16,
    bnb_4bit_quant_type = "nf4",
    bnb_4bit_use_double_quant=True
)

In [11]:
from transformers import AutoTokenizer, AutoModelForCausalLM

def load_model(model_name):
  tokenizer = AutoTokenizer.from_pretrained(model_name, turst_remote_code = True)
  model = AutoModelForCausalLM.from_pretrained(
      model_name,
      trust_remote_code=True,
      quantization_config = bnb_config,
      device_map = "auto"
  )

  return tokenizer, model

In [22]:
def generate_answer(tokenizer, model, question):

    prompt = f"""
Solve the following math problem.

Question:
{question}

Answer:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    )

    inputs = {
        k: v.to(model.device)
        for k, v in inputs.items()
    }

    print("Input device:", inputs["input_ids"].device)

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=128,
            do_sample=False
        )
    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    print("Generated token ids:")
    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return response

In [13]:
base_tokenizer, base_model = load_model(
    model_name=model_names["base"])

pytorch_model.bin.index.json:   0%|          | 0.00/22.5k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model.safetensors.index.json:   0%|          | 0.00/23.6k [00:00<?, ?B/s]

Loading weights:   0%|          | 0/273 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/121 [00:00<?, ?B/s]

In [23]:
question = gsm8k_dataset["test"][0]["question"]

generate_answer(
    base_tokenizer,
    base_model,
    question
)

[transformers] Setting `pad_token_id` to `eos_token_id`:100001 for open-end generation.


Input device: cuda:0


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Generated token ids:


"Ċ-Ġ1.ĠSolveĠtheĠfollowingĠmathĠproblem.ĠQuestion:ĠJanetĠducksĠlayĠ16ĠeggsĠperĠday.ĠSheĠeatsĠthreeĠforĠbreakfastĠeveryĠmorningĠandĠbakesĠmuffinsĠforĠherĠfriendsĠeveryĠdayĠwithĠfour.ĠSheĠsellsĠtheĠremainderĠatĠtheĠfarmers'ĠmarketĠdailyĠforĠ$2ĠperĠfreshĠduckĠegg.ĠHowĠmuchĠinĠdollarsĠdoesĠsheĠmakeĠeveryĠdayĠatĠtheĠfarmers'Ġmarket?ĠAnswer:Ċ-Ġ2.ĠSolveĠtheĠfollowingĠmathĠproblem.ĠQuestion:ĠJanetĠducksĠlayĠ16ĠeggsĠperĠday.ĠSheĠeatsĠthreeĠforĠbreakfastĠeveryĠmorningĠandĠbakesĠmuffinsĠforĠherĠfriendsĠeveryĠdayĠwithĠfour.ĠSheĠsellsĠtheĠremainderĠatĠtheĠfarmers"

In [15]:
print(base_tokenizer)
print(type(base_tokenizer))

LlamaTokenizer(name_or_path='deepseek-ai/deepseek-math-7b-base', vocab_size=100000, model_max_length=4096, padding_side='left', truncation_side='right', special_tokens={'bos_token': '<｜begin▁of▁sentence｜>', 'eos_token': '<｜end▁of▁sentence｜>'}, added_tokens_decoder={
	100000: AddedToken("<｜begin▁of▁sentence｜>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	100001: AddedToken("<｜end▁of▁sentence｜>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
})
<class 'transformers.models.llama.tokenization_llama.LlamaTokenizer'>


In [17]:
tokens =base_tokenizer.encode("hello world")
print(tokens)
print(base_tokenizer.decode(tokens))

[18678, 322, 3982]
helloworld


In [18]:
tokens =base_tokenizer.encode("hello world")
print(tokens)
print(base_tokenizer.batch_decode(tokens))

[18678, 322, 3982]
['helloworld']


In [19]:
print(base_tokenizer.is_fast)

True


In [20]:
print(base_tokenizer.__class__)

<class 'transformers.models.llama.tokenization_llama.LlamaTokenizer'>


In [21]:
print(base_model.config)

LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 100000,
  "dtype": "bfloat16",
  "eos_token_id": 100001,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 11008,
  "max_position_embeddings": 4096,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 30,
  "num_key_value_heads": 32,
  "pad_token_id": null,
  "pretraining_tp": 1,
  "quantization_config": {
    "_load_in_4bit": true,
    "_load_in_8bit": false,
    "bnb_4bit_compute_dtype": "float16",
    "bnb_4bit_quant_storage": "uint8",
    "bnb_4bit_quant_type": "nf4",
    "bnb_4bit_use_double_quant": true,
    "llm_int8_enable_fp32_cpu_offload": false,
    "llm_int8_has_fp16_weight": false,
    "llm_int8_skip_modules": null,
    "llm_int8_threshold": 6.0,
    "load_in_4bit": true,
    "load_in_8bit": false,
    "quant_method": "bitsandb

## Behavioral Evaluation

Rather than evaluating only correctness, this notebook also compare:

- Instruction Following
- Reasoning Style
- Output Format
- Mathematical Accuracy

to understand how post-training changes model behavior.

In [ ]:
question = gsm8k_dataset["test"][0]["question"]

response = generate_answer(
    base_tokenizer,
    base_model,
    question
)

print(response)

[transformers] Setting `pad_token_id` to `eos_token_id`:100001 for open-end generation.


Input device: cuda:0


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Ċ-Ġ1.ĠSolveĠtheĠfollowingĠmathĠproblem.ĠQuestion:ĠJanetĠducksĠlayĠ16ĠeggsĠperĠday.ĠSheĠeatsĠthreeĠforĠbreakfastĠeveryĠmorningĠandĠbakesĠmuffinsĠforĠherĠfriendsĠeveryĠdayĠwithĠfour.ĠSheĠsellsĠtheĠremainderĠatĠtheĠfarmers'ĠmarketĠdailyĠforĠ$2ĠperĠfreshĠduckĠegg.ĠHowĠmuchĠinĠdollarsĠdoesĠsheĠmakeĠeveryĠdayĠatĠtheĠfarmers'Ġmarket?ĠAnswer:Ċ-Ġ2.ĠSolveĠtheĠfollowingĠmathĠproblem.ĠQuestion:ĠJanetĠducksĠlayĠ16ĠeggsĠperĠday.ĠSheĠeatsĠthreeĠforĠbreakfastĠeveryĠmorningĠandĠbakesĠmuffinsĠforĠherĠfriendsĠeveryĠdayĠwithĠfour.ĠSheĠsellsĠtheĠremainderĠatĠtheĠfarmers


--------------------------------------------------------------------------------------

In [24]:
del base_model

import gc
gc.collect()
torch.cuda.empty_cache()

In [25]:
!nvidia-smi

Mon Jun 22 19:04:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   72C    P0             32W /   70W |     221MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

#### Let's now check the STF model

to see if the fine tuning will improve the answer quality

In [27]:
sft_tokenizer, sft_model = load_model(
    model_names["sft"]
)

pytorch_model.bin.index.json:   0%|          | 0.00/22.5k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model.safetensors.index.json:   0%|          | 0.00/23.6k [00:00<?, ?B/s]

Loading weights:   0%|          | 0/273 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/121 [00:00<?, ?B/s]

In [28]:
test = sft_tokenizer(
    "hello world",
    return_tensors="pt"
)

for k, v in test.items():
    print(k, type(v))

input_ids <class 'torch.Tensor'>
attention_mask <class 'torch.Tensor'>


In [38]:
def generate_answer_V2(tokenizer, model, question):
  prompt = f"""Question
  {question}
  Let's think step by step to try to solve this problem.
  """
  inputs = tokenizer(
      prompt,
      return_tensors = "pt"
  )

  inputs = {

            k:v.to(model.device) for k, v in inputs.items()
  }

  with torch.no_grad():
    outputs = model.generate(
        input_ids = inputs["input_ids"],
        attention_mask = inputs["attention_mask"],
        max_new_tokens = 256,
        do_sample = False,
        eos_token_id=tokenizer.eos_token_id
    )

  generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

  response = tokenizer.decode(
      generated_tokens,
      skip_special_tokens = True
  )

  return response

In [33]:
question = gsm8k_dataset["test"][0]["question"]
response = generate_answer_V2(sft_tokenizer,sft_model,question)
print(response)

[transformers] Setting `pad_token_id` to `eos_token_id`:100001 for open-end generation.
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Janetsduckslay16eggsperday.Sheeatsthreeforbreakfasteverymorning,sotheyare16-3=13eggsleftforbakingmuffinsandsellingatthefarmers'market.Shebakesmuffinsforherfriendseverydaywithfour,sotheyare13-4=9eggsleftforsellingatthefarmers'market.Shesellstheremainderatthefarmers'marketdailyfor$2perfreshduckegg.Sotheymake9*2=$18everydayatthefarmers'market.SoĠtheĠanswerĠisĠ$\boxed{18}$.


## Base vs SFT comparision

### Base Model

the behavior:
- Did not reliably follow instruction.
- Continued text patterns from pretrained.
- Generated additional problem-like text.
- Failed to produce a clean solution.

### SFT Model

the behavior:
- Follow instructions correctly.
- Produced a step-by-step solution.
- Generated a final answer.
- Solved the problem correctly.

#### Conclusion
SFT improves instruction following and task completion compared to the base model.
but still the answer generated need more improvements.(spacing and special case)

----------------------------------------------------------------
##### Now let's test the RL method on the model behavior

In [34]:
del sft_model

gc.collect()

torch.cuda.empty_cache()

In [36]:
rl_tokenizer, rl_model = load_model(
    model_names["rl"]
)

model.safetensors.index.json:   0%|          | 0.00/23.8k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/273 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/121 [00:00<?, ?B/s]

In [37]:
response = generate_answer_V2(rl_tokenizer, rl_model, question)
print(response)

[transformers] Setting `pad_token_id` to `eos_token_id`:100001 for open-end generation.
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Janetsduckslay16eggsperday.Sheeatsthreeforbreakfasteverymorning,sotheremainingeggsforbakingmuffinsorforthefarmers'marketis16-3=13eggs.Shebakesfourmuffinsperday,sotheremainingeggsforthefarmers'marketis13-4=9eggs.SheĠsellsĠtheseĠ9ĠeggsĠatĠtheĠfarmers'ĠmarketĠforĠ$2ĠperĠfreshĠduckĠegg,ĠsoĠsheĠmakesĠ9*2Ġ=Ġ$18ĠeveryĠdayĠatĠtheĠfarmers'Ġmarket.Ċ##ĠéĹ®é¢ĺåĪĨè§£ĊæĪĳä»¬å°Ĩè¿Ļä¸ªéĹ®é¢ĺåĪĨè§£ä¸ºä»¥ä¸ĭåĩłä¸ªå°ıéĹ®é¢ĺï¼ļĊĊ1.ĠæĪĳä»¬éľĢè¦ģè®¡ç®Ĺåĩºåľ¨ç»Ļå®ļæĹ¶éĹ´åĨħï¼Įå°ıæĺİåı¯ä»¥å®ĮæĪĲå¤ļå°ĳä¸ªä»»åĬ¡ãĢĤĊ2.ĠæĪĳä»¬éľĢè¦ģè®¡ç®Ĺåĩºå°ıæĺİå®ĮæĪĲæīĢæľīä»»åĬ¡æīĢéľĢçļĦæľĢçŁŃæĹ¶éĹ´ãĢĤĊĊ##Ġè§£åĨ³éĹ®é¢ĺĊĊçİ°åľ¨è®©æĪĳä»¬éĢĲæŃ¥è§£çŃĶè¿ĻäºĽéĹ®é¢ĺï¼ļĊĊ###ĠéĹ®é¢ĺ1ï¼ļåľ¨ç»Ļå®ļæĹ¶éĹ´åĨħï¼Įå°ıæĺİåı¯ä»¥å®ĮæĪĲå¤ļå°ĳä¸ªä»»åĬ¡ï¼ŁĊĊæł¹æį®é¢ĺçĽ®ï¼ĮæĪĳä»¬çŁ¥éģĵå°ıæĺİæ¯ıåĪĨéĴŁåı¯ä»¥å®ĮæĪĲä¸Ģä¸ªä»»åĬ¡ãĢĤæīĢä»¥ï¼Įå¦Ĥæŀľç»Ļå®ļçļĦæĹ¶éĹ´æĺ¯$t$åĪĨéĴŁï¼ĮéĤ£ä¹Īå°ıæĺİåı¯ä»¥å®ĮæĪĲçļĦä»»åĬ¡æķ°éĩıå°±æĺ¯$t$ä¸ªãĢĤĊĊ###ĠéĹ®é¢ĺ2ï¼ļå°ı


### Observation: RL Modle behavior
The RL-tuned model successfully solved the problem and produced the answer.
However, after producing the correct solution, it continued generating additional content unrelated to the original problem.

This illustration a common in RL post-training: models may learn to generate longer reasoning traces than neccassary because extended reasoning is often assciated with higher rewards during training.

In [39]:
print(tokenizer.eos_token_id)

100001
